# cPOS5-overexpression when grown in Glc-MeOH (60%-40%)

In [1]:
import cobra
import escher
import pandas as pd
import numpy as np
import json
import csv
from cobra.io import save_json_model
from cobra import Reaction
from cobra.flux_analysis.loopless import add_loopless, loopless_solution

In [2]:
print(cobra.__version__)

0.30.0


In [3]:
# model_path = 'iMT1026v3.xml'
model_path ='..\model\iMT1026v3jup.xml'

model = cobra.io.read_sbml_model(model_path)

model

Set parameter Username
Academic license - for non-commercial use only - expires 2026-10-13


Name,iMT1026v3
Memory address,2cd72af3610
Number of metabolites,1706
Number of reactions,2237
Number of genes,1026
Number of groups,77
Objective expression,1.0*Ex_biomass - 1.0*Ex_biomass_reverse_5354f
Compartments,"Vacuole, Cytosol, Mitochondria, Peroxisome, Extracellular space, Endoplasmic Reticulum, Golgi Apparatus, Nucleus, Mitochondrial intermembrane space"


In [4]:
model.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
glyc_e,Ex_glyc,0.9538,3,100.00%
nh4_e,Ex_nh4,0.3009,0,0.00%
o2_e,Ex_o2,1.616,0,0.00%
pi_e,Ex_pi,0.0122,0,0.00%
so4_e,Ex_so4,0.002128,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
biomass_e,Ex_biomass,-0.04851,0,0.00%
co2_e,Ex_co2,-1.163,1,100.00%
h_e,Ex_h,-0.3153,0,0.00%
h2o_e,Ex_h2o,-2.735,0,0.00%


In [5]:
# Change grwoth on glycerol to growth on 60% glucose and 40% methanol
# Add reaction describing this growth

biomass_gly = model.reactions.get_by_id('BIOMASS_glyc')
biomass_gly.bounds = (0,0)
biomass_gly

carbs = model.metabolites.get_by_id('CARBOHYDRATES_c')
DNA = model.metabolites.get_by_id('DNA_c')
lipids = model.metabolites.get_by_id('LIPIDS_c')
prot = model.metabolites.get_by_id('PROTEIN_c')
RNA = model.metabolites.get_by_id('RNA_c')
atp = model.metabolites.get_by_id('atp_c')
cof = model.metabolites.get_by_id('cof_c')
h2o = model.metabolites.get_by_id('h2o_c')
adp = model.metabolites.get_by_id('adp_c')
biomass = model.metabolites.get_by_id('biomass_c')
h = model.metabolites.get_by_id('h_c')
pi = model.metabolites.get_by_id('pi_c')

biomass_glc_meoh = Reaction ('biomass_glc_meoh_60/40')
biomass_glc_meoh.name = 'Biomass composition (g/g) - 60/40 Glucose/Methanol'
biomass_glc_meoh.add_metabolites({carbs: -0.33,
                                   DNA: -0.001,
                                   lipids: -0.042,
                                   prot: -0.49,
                                   RNA: -0.058,
                                   atp: -63.85,
                                   cof: -1,
                                   h2o: -63.85,
                                   adp: 63.85,
                                   pi: 63.85,
                                   biomass: 1,
                                   h: 63.85})
biomass_glc_meoh

Reaction identifier,biomass_glc_meoh_60/40
Name,Biomass composition (g/g) - 60/40 Glucose/Methanol
Memory address,0x260ad970a48
Stoichiometry,0.33 CARBOHYDRATES_c + 0.001 DNA_c + 0.042 LIPIDS_c + 0.49 PROTEIN_c + 0.058 RNA_c + 63.85 atp_c + cof_c + 63.85 h2o_c --> 63.85 adp_c + biomass_c + 63.85 h_c + 63.85 pi_c 0.33 Carbohydrates + 0.001 DNA + 0.042 Lipids + 0.49 PROTEIN + 0.058 RNA + 63.85 ATP + Cofactors and small molecules + 63.85 H2O --> 63.85 ADP + Biomass + 63.85 H+ + 63.85 Phosphate
GPR,
Lower bound,0.0
Upper bound,1000.0


In [6]:
# Add new biomass reaction to the model

print (len(model.reactions))
model.add_reactions([biomass_glc_meoh])
print (len(model.reactions))

2237
2238


In [7]:
#Check the addition of the biomass reaction to the model
BIOMASS_GLCMEOH = model.reactions.get_by_id('biomass_glc_meoh_60/40')
BIOMASS_GLCMEOH

Reaction identifier,biomass_glc_meoh_60/40
Name,Biomass composition (g/g) - 60/40 Glucose/Methanol
Memory address,0x260ad970a48
Stoichiometry,0.33 CARBOHYDRATES_c + 0.001 DNA_c + 0.042 LIPIDS_c + 0.49 PROTEIN_c + 0.058 RNA_c + 63.85 atp_c + cof_c + 63.85 h2o_c --> 63.85 adp_c + biomass_c + 63.85 h_c + 63.85 pi_c 0.33 Carbohydrates + 0.001 DNA + 0.042 Lipids + 0.49 PROTEIN + 0.058 RNA + 63.85 ATP + Cofactors and small molecules + 63.85 H2O --> 63.85 ADP + Biomass + 63.85 H+ + 63.85 Phosphate
GPR,
Lower bound,0.0
Upper bound,1000.0


In [8]:
#Remove glycerol supply
glycerol_exchange = model.exchanges.get_by_id('Ex_glyc')
glycerol_exchange.bounds = (0,0)
glycerol_exchange

Reaction identifier,Ex_glyc
Name,Glycerol exchange
Memory address,0x260acf7b608
Stoichiometry,glyc_e --> Glycerol -->
GPR,
Lower bound,0
Upper bound,0


In [9]:
# Add qmeoh constraints observed in chemostat cultivations from Sergi Monforte's doctoral thesis
methanol_exchange = model.exchanges.get_by_id('Ex_meoh')
methanol_exchange.bounds = (-1.24, -1.18)
methanol_exchange

Reaction identifier,Ex_meoh
Name,Methanol exchange
Memory address,0x260acfbe208
Stoichiometry,meoh_e <-- Methanol <--
GPR,
Lower bound,-1.24
Upper bound,-1.18


In [10]:
# Add qgluc constraints observed in chemostat cultivations from Sergi Monforte's doctoral thesis
glucose_exchange = model.reactions.get_by_id('Ex_glc_D')
glucose_exchange.bounds = (-0.72, -0.7)
glucose_exchange

Reaction identifier,Ex_glc_D
Name,D-Glucose exchange
Memory address,0x260acfa9a08
Stoichiometry,glc_D_e <-- D-Glucose <--
GPR,
Lower bound,-0.72
Upper bound,-0.7


In [11]:
# Change the reactions for the synthesis of lipids, proteins and sterols from glycerol to those from glucose

model.reactions.get_by_id('LIPIDS_glyc').bounds = (0,0)
model.reactions.get_by_id('PROTEINS_glyc').bounds = (0,0)
model.reactions.get_by_id('STEROLS_glyc').bounds = (0,0)

# note: these reactions from glucose do not have the glucose specification
model.reactions.get_by_id('LIPIDS').bounds = (0,1000)
model.reactions.get_by_id('PROTEINS').bounds = (0,1000)
model.reactions.get_by_id('STEROLS').bounds = (0,1000)

In [12]:
# ATP maintenance requirement (NGAME = Non-Growth Associated Maintenance Energy) based on previous studies on 
# the growth of X33-ROL on Gluc/MeOH from the group (Eric's Master Thesis)

model.reactions.get_by_id('ATPM').bounds = (1.96, 1000)

In [13]:
rolAA = model.reactions.get_by_id('rolAA')
rolAA.bounds = (0,1000)
rolRNA = model.reactions.get_by_id('rolRNA')
rolRNA.bounds = (0,1000)
rolDNA = model.reactions.get_by_id('rolDNA')
rolDNA.bounds = (0,1000)
pROL = model.reactions.get_by_id('pROL')
pROL.bounds = (0,1000)
Rol_transport =  model.reactions.get_by_id('ROLt')
Rol_transport.bounds = (0,1000)
ROL_exchange = model.exchanges.get_by_id('Ex_rol')
ROL_exchange.bounds = (0.003,1000) #Value obtained from calculations of data from previous studies in our research group

pFAB = model.reactions.get_by_id('pFAB')
pFAB.bounds = (0,0)
fabAA = model.reactions.get_by_id('fabAA')
fabAA.bounds = (0,0)
fabt = model.reactions.get_by_id('fabt')
fabt.bounds = (0,0)
fabRNA = model.reactions.get_by_id('fabRNA')
fabRNA.bounds = (0,0)
fabDNA = model.reactions.get_by_id('fabDNA')
FAB_exchange = model.exchanges.get_by_id('Ex_fab')
FAB_exchange.bounds = (0,0)


# Extra reactions that must be closed for simulations to run smoothly:

APAT2r = model.reactions.get_by_id('APAT2r')
APAT2r.bounds = (0,0) # reaction not present in Pichia, it is yet to be removed from the model

MMSAD3 = model.reactions.get_by_id('MMSAD3')
MMSAD3.bounds = (0,0) # The reduction reaction of MSA into Acetil-CoA it is due to an unspecific effect. Reaction
# under evaluation of being kept or not.

In [14]:
#Creating the cPOS5 reaction
atp_c = model.metabolites.get_by_id('atp_c')
nadh_c = model.metabolites.get_by_id('nadh_c')
adp_c = model.metabolites.get_by_id('adp_c')
nadph_c = model.metabolites.get_by_id('nadph_c')
h_c = model.metabolites.get_by_id('h_c')

cPOS5 = Reaction('cPOS5')
cPOS5.name = 'Cytosolic NADH Kinase from Saccharomyces cerevisiae'
cPOS5.add_metabolites({atp_c: -1, 
                           nadh_c: -1, 
                           adp_c: 1, 
                           nadph_c: 1,
                               h_c: 1})
cPOS5.bounds = (0,0)
print (cPOS5.reaction)
cPOS5

# from Saccharomyces. Found in Brenda ('for references in articles please use BRENDA:EC2.7.1.86')
# Also in Uniprot with ID Q06892 and ID YPL188W (Saccharomyces Genome Database)

atp_c + nadh_c --> adp_c + h_c + nadph_c


Reaction identifier,cPOS5
Name,Cytosolic NADH Kinase from Saccharomyces cerevisiae
Memory address,0x260ad9509c8
Stoichiometry,atp_c + nadh_c --> adp_c + h_c + nadph_c ATP + NADH --> ADP + H+ + NADPH
GPR,
Lower bound,0
Upper bound,0


In [15]:
#Add cPOS5 reaction to the model
print (len(model.reactions))
model.add_reactions([cPOS5])
print (len(model.reactions))

2238
2239


In [16]:
nadph_c = model.metabolites.get_by_id('nadph_c')
nadh_c = model.metabolites.get_by_id('nadh_c')
nad_c = model.metabolites.get_by_id('nad_c')
nadp_c = model.metabolites.get_by_id('nadp_c')

In [17]:
nadph_m = model.metabolites.get_by_id('nadph_m')
nadh_m = model.metabolites.get_by_id('nadh_m')
nad_m = model.metabolites.get_by_id('nad_m')
nadp_m = model.metabolites.get_by_id('nadp_m')

## Reaction Ratios as constraints

### The experimental data was extracted from the paper:
#### "Metabolic flux analysis of recombinant Pichia pastoris growing on different glycerol/methanol mixtures by iterative fitting of NMR-derived 13C labelling data from proteinogenic amino acids" by Joel Jordà, 2014

In [18]:
ArabitolSecretion = model.exchanges.get_by_id('Ex_abt_D')
ICL_Reaction = model.reactions.get_by_id('ICLx')
FBA_Reaction = model.reactions.get_by_id('FBA')
MAE2m_Reaction = model.reactions.get_by_id('ME2m')
MAE1m_Reaction = model.reactions.get_by_id('ME1m')
MAE1_Reaction = model.reactions.get_by_id('ME1')
MALSp_Reaction = model.reactions.get_by_id('MALSp')
CSm_Reaction = model.reactions.get_by_id('CSm')
ALCD19_Reaction = model.reactions.get_by_id('ALCD19')
MDHm_Reaction = model.reactions.get_by_id('MDHm')
PDH_Reaction = model.reactions.get_by_id('PDHcm')
NADHD_Reaction = model.reactions.get_by_id('NADH2_u6m')

In [19]:
ReactionRatio1 = model.problem.Constraint(model.reactions.CSm.flux_expression - model.reactions.ACONTm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio1)

ReactionRatio2 = model.problem.Constraint(model.reactions.AKGDam.flux_expression - model.reactions.FUMm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio2)

ReactionRatio8 = model.problem.Constraint(68*model.reactions.AKGDam.flux_expression - 46*model.reactions.CSm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio8)

ReactionRatio9 = model.problem.Constraint(model.reactions.G6PDH2.flux_expression - model.reactions.GND.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio9)

ReactionRatio10 = model.problem.Constraint(0.8*model.reactions.MDHm.flux_expression - model.reactions.FUMm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio10)

ReactionRatio12 = model.problem.Constraint(35*model.reactions.PYK.flux_expression - 143*model.reactions.PC.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio12)

ReactionRatio13 = model.problem.Constraint(68*model.reactions.PYK.flux_expression - 143*model.reactions.PYRt2m.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio13)


ReactionRatio14 = model.problem.Constraint(40*model.reactions.GAPD.flux_expression - 145*model.reactions.FBA.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio14)


ReactionRatio16 = model.problem.Constraint(model.reactions.GAPD.flux_expression - model.reactions.PYK.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio16)

ReactionRatio20 = model.problem.Constraint(0.18*model.reactions.FALDtx.flux_expression - 0.82*model.reactions.DAS.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio20)

ReactionRatio21 = model.problem.Constraint(model.reactions.G6PDH2.flux_expression + 0.4*model.reactions.Ex_glc_D.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio21)

### pFBA Simulation for Reference WT

In [20]:
#WT Rol producer strain reference flux distribution
pfba_WT = cobra.flux_analysis.pfba(model)

In [21]:
pfba_WT.fluxes['Ex_rol']

0.003

In [22]:
pfba_WT.fluxes['Ex_biomass']

0.08043226373244569

In [23]:
pfba_WT.fluxes['Ex_co2']

2.362878923397944

In [24]:
pfba_WT.fluxes['Ex_o2']

-2.893938162906779

In [25]:
pfba_WT.fluxes['Ex_glc_D']

-0.72

In [26]:
# NADPH turnover rate WT strain

positive_nadph_flux_sum_WT = 0.0
for reaction in nadph_c.reactions:
    flux = pfba_WT.fluxes[reaction.id]
    coef = reaction.get_coefficient(nadph_c)
    # No totes les reaccions tenen 1 de coeficient estequiomètric pel cofactor
    #independentment si s'està produint o consumint
    positive_nadph_flux_sum_WT += abs(flux*coef)
    

print (positive_nadph_flux_sum_WT/2)

0.8232318305171787


In [27]:
#NADH turnover rate WT strain

positive_nadh_flux_sum_WT = 0.0
for reaction in nadh_c.reactions:
    flux = pfba_WT.fluxes[reaction.id]
    coef = reaction.get_coefficient(nadh_c)
    positive_nadh_flux_sum_WT += abs(flux*coef)

print (positive_nadh_flux_sum_WT/2)

3.137624517906749


In [28]:
# NADP turnover rate WT strain

positive_nadp_flux_sum_WT = 0.0
for reaction in nadp_c.reactions:
    flux = pfba_WT.fluxes[reaction.id]
    coef = reaction.get_coefficient(nadp_c)
    positive_nadp_flux_sum_WT += abs(flux*coef)

print (positive_nadp_flux_sum_WT/2)

0.8232319109494423


In [29]:
# NAD turnover rate WT strain

positive_nad_flux_sum_WT = 0.0
for reaction in nad_c.reactions:
    flux = pfba_WT.fluxes[reaction.id]
    coef = reaction.get_coefficient(nad_c)
    positive_nad_flux_sum_WT += abs(flux*coef)

print (positive_nad_flux_sum_WT/2)

3.1376246787712763


In [30]:
#NADH mitochondrial turnover rate WT strain

positive_nadh_flux_sum_WT = 0.0
for reaction in nadh_m.reactions:
    flux = pfba_WT.fluxes[reaction.id]
    coef = reaction.get_coefficient(nadh_m)
    positive_nadh_flux_sum_WT += abs(flux*coef)

print (positive_nadh_flux_sum_WT/2)

1.141258256213425


In [31]:
# NAD mitochondrial turnover rate WT strain

positive_nad_flux_sum_WT = 0.0
for reaction in nad_m.reactions:
    flux = pfba_WT.fluxes[reaction.id]
    coef = reaction.get_coefficient(nad_m)
    positive_nad_flux_sum_WT += abs(flux*coef)

print (positive_nad_flux_sum_WT/2)

1.141258256213425


In [32]:
# ATP turnover rate WT strain

positive_atp_flux_sum_WT = 0.0
for reaction in atp_c.reactions:
    flux = pfba_WT.fluxes[reaction.id]
    coef = reaction.get_coefficient(atp_c)
    positive_atp_flux_sum_WT += abs(flux*coef)

print (positive_atp_flux_sum_WT/2)

9.966756438771625


## MOMA Simulations

### Carbon Source: Glc/MeOH

### Overexpressed gene --> POS5

In [33]:
# Remove Reaction Ratio Constraints before starting MOMA simulations

ReactionRatioList = [ReactionRatio1, ReactionRatio2, ReactionRatio8, ReactionRatio9, ReactionRatio10,
                     ReactionRatio12, ReactionRatio13, ReactionRatio14, ReactionRatio16, ReactionRatio20, ReactionRatio21]
                    

model.remove_cons_vars(ReactionRatioList)

In [34]:
# As initially the flux value of the ectopic enzyme is not known many range values had to be explored:

cPOS5range = (0.02, 0.04, 0.06, 0.08, 0.1, 0.12, 0.14, 0.16, 0.18, 0.2)
cPOS5range1 = (0.002,0.004,0.006,0.008,0.01,0.012,0.014,0.016, 0.018, 0.02)
cPOS5range2 = (0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1)
cPOS5range3 = (0.001,0.002,0.003,0.004,0.005,0.006,0.007,0.008)
Rolrangelow = (0.0004, 0.00042, 0.00045, 0.00047, 0.00049, 0.00051, 0.00054, 0.00056, 0.00058, 0.0006)

In [37]:
#MOMA non-linear Simulations Glucose 60 MeOH 40 Overexpressing POS5
MomaResults = []
for x in cPOS5range:
        cPOS5.bounds = (x, x)
        
        # Perform MOMA
        moma_result = cobra.flux_analysis.moma(model,pfba_WT,0) #Boolean per si el MOMA es lineal o no
        MomaResults.append(moma_result.fluxes)
                 
        print(cPOS5.bounds, moma_result.fluxes['Ex_rol'], moma_result.fluxes['Ex_biomass'],  moma_result.fluxes['Ex_o2'],  moma_result.fluxes['Ex_co2'])
        

(0.02, 0.02) 0.003227865646250655 0.07997046876221059 -2.8942219967683935 2.3630444387633434
(0.04, 0.04) 0.0034643010707745754 0.07948676845076183 -2.8946511524091143 2.3633063969952763
(0.06, 0.06) 0.0037022034754247905 0.07899960839463649 -2.8951039110888206 2.3635853996382172
(0.08, 0.08) 0.0039400615659208995 0.07851246054059431 -2.8955565670475742 2.3638645451107463
(0.1, 0.1) 0.0041790620771673375 0.07802380951886372 -2.8960055561152633 2.364146922798062
(0.12, 0.12) 0.004419162938568663 0.07753307434006 -2.896456421785011 2.3644348977852885
(0.14, 0.14) 0.004659299561181353 0.0770406905847935 -2.896923675524058 2.364730466181266
(0.16, 0.16) 0.004902177669044707 0.07654067320038488 -2.897459478442969 2.3650685330118013
(0.18, 0.18) 0.005145089987609243 0.07603946954626864 -2.898017170430179 2.36541918001121
(0.2, 0.2) 0.005388093464151627 0.0755377618238241 -2.8985759069339476 2.365770051485307


In [38]:
#NonLinear MOMA Simulations Glucose 60 MeOH 40 Overexpressing POS5 Redox cofactors exploration
#Ara els càlculs tenen més sentit ja el valor de turnover de les formes oxidada i reduides del mateix
#cofactor son equivalents
MomaResults = []
for x in cPOS5range:
        cPOS5.bounds = (x, x)
        
        # Perform MOMA
        moma_result = cobra.flux_analysis.moma(model,pfba_WT,0)
        MomaResults.append(moma_result.fluxes)
        
        # Calculate the sum of positive fluxes involving nadph_c
        positive_nadph_flux_sum_moma = 0.0
        positive_nadh_flux_sum_moma = 0.0
        positive_nadp_flux_sum_moma = 0.0
        positive_nad_flux_sum_moma = 0.0
        
        for reaction in nadph_c.reactions:
            # Get the flux value for this reaction from the MOMA result
            nadph_flux_moma = moma_result.fluxes[reaction.id]
            coef1 = reaction.get_coefficient(nadph_c)
            
            # Add the flux to the sum if it's positive
            
            positive_nadph_flux_sum_moma += abs(nadph_flux_moma*coef1)
                
        for reaction in nadh_c.reactions:
            # Get the flux value for this reaction from the MOMA result
            nadh_flux_moma = moma_result.fluxes[reaction.id]
            coef2 = reaction.get_coefficient(nadh_c)
            # Add the flux to the sum if it is positive
            
            positive_nadh_flux_sum_moma += abs(nadh_flux_moma*coef2)
                
        for reaction in nadp_c.reactions:
            # Get the flux value for this reaction from the MOMA result
            nadp_flux_moma = moma_result.fluxes[reaction.id]
            coef3 = reaction.get_coefficient(nadp_c)
            # Add the flux to the sum if it's positive
            
            positive_nadp_flux_sum_moma += abs(nadp_flux_moma*coef3)
                
        for reaction in nad_c.reactions:
            # Get the flux value for this reaction from the MOMA result
            nad_flux_moma = moma_result.fluxes[reaction.id]
            coef4 = reaction.get_coefficient(nad_c)
            # Add the flux to the sum if it is positive
            positive_nad_flux_sum_moma += abs(nad_flux_moma*coef4)
        
        print(cPOS5.bounds, positive_nadph_flux_sum_moma/2, positive_nadh_flux_sum_moma/2, positive_nadp_flux_sum_moma/2, positive_nad_flux_sum_moma/2)

(0.02, 0.02) 0.8406293186215285 3.145967817887603 0.8406293186215288 3.1459678978580627
(0.04, 0.04) 0.8568786660648919 3.1580711571212974 0.856878666064895 3.1580712366080665
(0.06, 0.06) 0.8729413232516764 3.170783976739739 0.8729413232516758 3.1707840557393467
(0.08, 0.08) 0.8890038901977168 3.183496769104431 0.8890038901977162 3.183496847616892
(0.1, 0.1) 0.9049960641609643 3.1965811184593287 0.9049960641609645 3.1965811964831383
(0.12, 0.12) 0.920878738025198 3.2101403328829456 0.9208787380252073 3.210140410416021
(0.14, 0.14) 0.9369441568479729 3.223594468701497 0.9369441568479725 3.223594545742187
(0.16, 0.16) 0.9538916852808142 3.2370618809432226 0.9538916852808135 3.2370619574838977
(0.18, 0.18) 0.9707760784481603 3.2508192002182295 0.9707760784481605 3.2508192762576993
(0.2, 0.2) 0.987661015029572 3.264544098953317 0.9876610150295714 3.2645441744910784


In [40]:
#NonLinear MOMA Simulations Glucose 60 MeOH 40 Overexpressing POS5 Redox cofactors exploration (mitochondrial)
MomaResults = []
for x in cPOS5range:
        cPOS5.bounds = (x, x)
        
        # Perform MOMA
        moma_result = cobra.flux_analysis.moma(model,pfba_WT,0)
        MomaResults.append(moma_result.fluxes)
        
        # Calculate the sum of positive fluxes involving nadph_c
        positive_atp_flux_sum_moma = 0.0
        positive_nadh_flux_sum_moma = 0.0
        
        positive_nad_flux_sum_moma = 0.0
        
        for reaction in atp_c.reactions:
            # Get the flux value for this reaction from the MOMA result
            atp_flux_moma = moma_result.fluxes[reaction.id]
            coef1 = reaction.get_coefficient(atp_c)
            
            # Add the flux to the sum if it's positive
            positive_atp_flux_sum_moma += abs(atp_flux_moma*coef1)
                
        for reaction in nadh_m.reactions:
            # Get the flux value for this reaction from the MOMA result
            nadh_flux_moma = moma_result.fluxes[reaction.id]
            coef2 = reaction.get_coefficient(nadh_m)
            # Add the flux to the sum if it is positive
            positive_nadh_flux_sum_moma += abs(nadh_flux_moma*coef2)
                
        for reaction in nad_m.reactions:
            # Get the flux value for this reaction from the MOMA result
            nad_flux_moma = moma_result.fluxes[reaction.id]
            coef3 = reaction.get_coefficient(nad_m)
            # Add the flux to the sum if it is positive
            positive_nad_flux_sum_moma += abs(nad_flux_moma*coef3)
        
        print(cPOS5.bounds, positive_atp_flux_sum_moma/2, positive_nadh_flux_sum_moma/2, positive_nad_flux_sum_moma/2)

(0.02, 0.02) 9.967237059665873 1.1414939478447061 1.1414939478447061
(0.04, 0.04) 9.967746183403978 1.14187330290366 1.14187330290366
(0.06, 0.06) 9.968255070401637 1.142277765172474 1.142277765172474
(0.08, 0.08) 9.968763718789763 1.1426827920646194 1.1426827920646194
(0.1, 0.1) 9.969262178780703 1.1429922271342416 1.142992227134243
(0.12, 0.12) 9.969744495769328 1.143204663888674 1.143204663888674
(0.14, 0.14) 9.97024166736639 1.1434772131395303 1.1434772131395303
(0.16, 0.16) 9.97075394734843 1.143852922664115 1.143852922664115
(0.18, 0.18) 9.97126832940181 1.1442409744067492 1.1442409744067492
(0.2, 0.2) 9.971777419720004 1.144620061379535 1.1446200613795354


In [43]:
MMResults_POS5_002 = MomaResults[0]
MMResults_POS5_004 = MomaResults[1]
MMResults_POS5_006 = MomaResults[2]
MMResults_POS5_008 = MomaResults[3]
MMResults_POS5_01 = MomaResults[4]
MMResults_POS5_012 = MomaResults[5]
MMResults_POS5_014 = MomaResults[6]
MMResults_POS5_016 = MomaResults[7]
MMResults_POS5_018 = MomaResults[8]
MMResults_POS5_02 = MomaResults[9]

In [44]:
#Codi per guardar la raw data dels fluxos en totes les reaccions a Excel
MMResults_POS5_002.to_excel('POS5_NLMOMA_RRed_002.xlsx', sheet_name = '002')
MMResults_POS5_004.to_excel('POS5_NLMOMA_RRed_004.xlsx', sheet_name = '004')
MMResults_POS5_006.to_excel('POS5_NLMOMA_RRed_006.xlsx', sheet_name = '006')
MMResults_POS5_008.to_excel('POS5_NLMOMA_RRed_008.xlsx', sheet_name = '008')
MMResults_POS5_01.to_excel('POS5_NLMOMA_RRed_01.xlsx', sheet_name = '01')
MMResults_POS5_012.to_excel('POS5_NLMOMA_RRed_012.xlsx', sheet_name = '012')
MMResults_POS5_014.to_excel('POS5_NLMOMA_RRed_014.xlsx', sheet_name = '014')
MMResults_POS5_016.to_excel('POS5_NLMOMA_RRed_016.xlsx', sheet_name = '016')
MMResults_POS5_018.to_excel('POS5_NLMOMA_RRed_018.xlsx', sheet_name = '018')
MMResults_POS5_02.to_excel('POS5_NLMOMA_RRed_02.xlsx', sheet_name = '02')

In [40]:
pfba_WT.fluxes.to_excel('pFBARef_RRed_Glc60_MeOH40.xlsx', sheet_name = 'Ref')